# 🚢 Titanic: Omon Qolishni Bashorat Qilish

**Birinchi shaxsiy Machine Learning loyihasi — Classification (tasniflash)**

Bu loyihada Kaggle'ning mashhur ["Titanic - Machine Learning from Disaster"](https://www.kaggle.com/competitions/titanic) dataset'idan foydalanib, yo'lovchining Titanic falokatida omon qolgan yoki qolmaganini bashorat qiluvchi model quramiz.

## Loyiha maqsadi
- Ma'lumotlarni tozalash va tayyorlash (EDA + preprocessing)
- Tarixiy "Women and children first" qoidasini real data orqali tekshirish
- Ikkita classification modelini (Logistic Regression va Random Forest) qurish va solishtirish
- Feature engineering va hyperparameter tuning orqali modelni yaxshilashga urinish

## Dataset haqida
| Fayl | Tavsif |
|---|---|
| `train.csv` | 891 yo'lovchi, `Survived` ustuni bilan (model o'qitish uchun) |
| `test.csv` | 418 yo'lovchi, `Survived`siz (bashorat qilinadigan qism) |
| `gender_submission.csv` | Namunaviy submission formati |

**Ustunlar:** `PassengerId`, `Survived` (0=yo'q, 1=ha), `Pclass` (bilet klassi), `Name`, `Sex`, `Age`, `SibSp` (aka-uka/turmush o'rtoq soni), `Parch` (ota-ona/bola soni), `Ticket`, `Fare`, `Cabin`, `Embarked` (chiqqan port)

---


## 1. Kutubxonalar va Data'ni yuklash

Avval kerakli kutubxonalarni import qilamiz va Google Drive'dan CSV fayllarni o'qib olamiz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = '/content/drive/MyDrive/ML_Projects/titanic'

train = pd.read_csv(f'{path}/train.csv')
test = pd.read_csv(f'{path}/test.csv')

train.head()

## 2. Dastlabki tahlil (EDA — Exploratory Data Analysis)

Modelni qurishdan oldin data qanday ko'rinishda ekanini tushunish kerak: qancha qator/ustun bor, qaysi ustunlarda ma'lumot yetishmayapti, raqamli ustunlarning statistikasi qanday.

In [ ]:
train.info()

**Natija:** 891 qator, 12 ustun. `Age`, `Cabin`, `Embarked` ustunlarida ba'zi qiymatlar yo'q (`non-null count` umumiy 891 dan kam).

In [ ]:
train.isnull().sum()

**Missing values xulosasi:**
- `Age` — 177 ta yo'q (~20%) — muhim feature, to'ldirish kerak
- `Cabin` — 687 ta yo'q (~77%) — deyarli bo'sh, to'g'ridan-to'g'ri ishlatib bo'lmaydi
- `Embarked` — 2 ta yo'q — arzimas, oson to'ldiriladi

In [ ]:
train.describe()

**Statistikadan xulosalar:**
- `Survived` o'rtachasi 0.38 → yo'lovchilarning ~38% omon qolgan, ~62% halok bo'lgan
- `Age` o'rtacha 29.7 yosh, eng kichigi 0.42 (chaqaloq), eng kattasi 80
- `Fare` (bilet narxi) da katta tarqoqlik bor: o'rtacha ~32, lekin maksimum 512 — bu outlier (chetga chiqib turgan qiymat) borligini ko'rsatadi

---


## 3. "Women and Children First" — tarixiy qoidani data orqali tekshirish

Titanic falokatida qutqaruv qayiqlariga birinchi navbatda ayollar va bolalarni o'tqazish qoidasi amal qilgan degan tarixiy ma'lumot bor. Keling, buni real statistika bilan tasdiqlaymiz.

In [ ]:
# Jins bo'yicha omon qolish foizi
survival_by_sex = train.groupby('Sex')['Survived'].mean() * 100
print("Jins bo'yicha omon qolish foizi:")
print(survival_by_sex)

plt.figure(figsize=(6, 4))
sns.barplot(x=survival_by_sex.index, y=survival_by_sex.values, palette=['#4C72B0', '#DD8452'])
plt.ylabel('Omon qolish foizi (%)')
plt.title('Jins bo\'yicha omon qolish foizi')
plt.ylim(0, 100)
plt.show()

**Kutilgan natija:** ayollarning omon qolish foizi (~74%) erkaklarnikidan (~19%) sezilarli darajada yuqori chiqadi — bu "women first" qoidasining aniq tasdig'i.

In [ ]:
# Yosh bo'yicha guruhlash: bola (<16 yosh) va kattalar
train['AgeGroup'] = np.where(train['Age'] < 16, 'Bola (<16)', 'Kattalar (16+)')

survival_by_age = train.groupby('AgeGroup')['Survived'].mean() * 100
print("Yosh guruhi bo'yicha omon qolish foizi:")
print(survival_by_age)

plt.figure(figsize=(6, 4))
sns.barplot(x=survival_by_age.index, y=survival_by_age.values, palette=['#55A868', '#C44E52'])
plt.ylabel('Omon qolish foizi (%)')
plt.title('Yosh guruhi bo\'yicha omon qolish foizi')
plt.ylim(0, 100)
plt.show()

In [ ]:
# Ikkalasini birga: jins + yosh guruhi bo'yicha omon qolish
combo = train.groupby(['Sex', 'AgeGroup'])['Survived'].mean().unstack() * 100
print(combo)

combo.plot(kind='bar', figsize=(7, 5))
plt.ylabel('Omon qolish foizi (%)')
plt.title('Jins va yosh guruhi bo\'yicha omon qolish foizi')
plt.xticks(rotation=0)
plt.legend(title='Yosh guruhi')
plt.show()

**Xulosa:** Grafiklar aniq ko'rsatadiki — ayollar va bolalar (ayniqsa qiz bolalar) eng yuqori omon qolish foiziga ega, kattalar erkaklar esa eng past foizga ega. Bu "Women and children first" tarixiy qoidasining data orqali tasdiqlangan dalili — va bu keyinchalik modelimizda `Sex` ustuni nima uchun eng muhim feature bo'lib chiqishini oldindan tushuntirib beradi.

`AgeGroup` ustunini vaqtinchalik tahlil uchun yaratdik, endi uni asosiy datadan olib tashlaymiz (model uchun keyinroq boshqacha feature'lar tayyorlaymiz).

In [ ]:
train = train.drop('AgeGroup', axis=1)

---


## 4. Ma'lumotlarni tozalash (Missing Values)

EDA bosqichida aniqlangan yo'q qiymatlarni to'ldiramiz:
- **`Age`** → median (o'rtacha qiymat emas — median outlier'larga chidamli, chunki `Fare`da ko'rganimizdek data'da chetga chiqib turgan qiymatlar bor)
- **`Embarked`** → mode (eng ko'p uchraydigan qiymat) — atigi 2 ta yo'q qiymat uchun bu yetarli
- **`Cabin`** → 77% bo'sh bo'lgani uchun to'ldirish mantiqsiz. Buning o'rniga "kabinasi ma'lummi yoki yo'qmi" degan yangi binary (0/1) feature yasaymiz — chunki kabina ma'lum bo'lishining o'zi (masalan yuqori klass yo'lovchilarda ko'proq) foydali signal bo'lishi mumkin

In [ ]:
train['Age'] = train['Age'].fillna(train['Age'].median())
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

train['Has_Cabin'] = train['Cabin'].notnull().astype(int)
train = train.drop('Cabin', axis=1)

train.isnull().sum()

## 5. Categorical ustunlarni raqamga o'tkazish (Encoding)

Model matn qiymatlar (`male`/`female`, `S`/`C`/`Q`) bilan ishlay olmaydi, shuning uchun ularni raqamga aylantiramiz:
- **`Sex`** — faqat 2 ta qiymat bor, shuning uchun oddiy Label Encoding (`male`→0, `female`→1) yetarli
- **`Embarked`** — 3 ta qiymat bor (`S`, `C`, `Q`), shuning uchun One-Hot Encoding qo'llaymiz. `drop_first=True` bitta ustunni (`Embarked_C`) tashlab yuboradi — bu "dummy variable trap"ning oldini oladi: `Embarked_Q=0` va `Embarked_S=0` bo'lsa, bu avtomatik ravishda `C`(Cherbourg)dan chiqqanini bildiradi, ma'lumot yo'qolmaydi

In [ ]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)

train.head()

## 6. Keraksiz ustunlarni olib tashlash

`PassengerId` — shunchaki indeks, bashoratga aloqasi yo'q. `Name` va `Ticket` — erkin matn, hozirgi shaklda modelga foyda bermaydi (keyinchalik ismdan unvon — Mr/Mrs/Miss — ajratib olish mumkin bo'lardi, lekin bu loyihada soddalik uchun olib tashlaymiz).

In [ ]:
train_model = train.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

train_model.info()

---


## 7. Baseline model: Logistic Regression

Data endi to'liq raqamli va tayyor. Datani train/validation qismlarga bo'lib (80%/20%), birinchi — eng oddiy va tushunarli classification modelidan boshlaymiz.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = train_model.drop('Survived', axis=1)
y = train_model['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

**Natija: ~82.1% accuracy.** Bu Titanic uchun yaxshi boshlang'ich natija (odatda 75-85% oraliq "sog'lom" hisoblanadi). `Class 0` (halok bo'lganlar) uchun precision/recall biroz yuqoriroq — bu data'da halok bo'lganlar ko'proq (62%) bo'lgani uchun mantiqiy.

## 8. Random Forest bilan solishtirish

Random Forest — bir nechta qaror daraxtlarini birlashtiruvchi kuchliroq algoritm, u nochiziqli bog'liqliklarni ham ushlay oladi.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred_rf))
print(classification_report(y_val, y_pred_rf))

**Natija: ~79.9% accuracy** — bu safar Logistic Regression'dan pastroq chiqdi. Sabab: dataset nisbatan kichik (891 qator) va oddiy, bunday holatda murakkab modellar ko'pincha ortiqcha (overfitting) bo'lib qoladi, oddiy chiziqli model esa yaxshiroq umumlashtiradi (generalize qiladi).

## 9. Feature Importance — qaysi ustun eng muhim?

Random Forest har bir feature bashoratga qanchalik hissa qo'shganini ko'rsata oladi.

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

plt.figure(figsize=(7, 5))
sns.barplot(x='importance', y='feature', data=importances, color='#4C72B0')
plt.title('Feature Importance (Random Forest)')
plt.show()

**Xulosa:** `Sex` eng muhim feature bo'lib chiqdi — bu 3-bo'limda ko'rgan "women and children first" tahlilimizni to'g'ridan-to'g'ri tasdiqlaydi. `Fare` va `Age` ham katta rol o'ynagan (yuqori bilet narxi → yaxshiroq joylashuv/imtiyoz → omon qolish ehtimoli yuqoriroq). `Embarked` esa deyarli ahamiyatsiz — port qayerdan chiqqanlik omon qolish bilan bevosita bog'liq emas, bu mantiqiy.

---


## 10. Feature Engineering: `FamilySize` va `IsAlone`

`SibSp` (aka-uka/turmush o'rtoq) va `Parch` (ota-ona/bola) alohida-alohida kam ma'lumot beradi. Ularni birlashtirib, yangi feature'lar yasaymiz:
- **`FamilySize`** — umumiy oila a'zolari soni (+1, chunki o'zi ham hisoblanadi)
- **`IsAlone`** — yo'lovchi yolg'iz sayohat qilyaptimi (binary)

Bu klassik Titanic feature engineering triki — ba'zan yaxshilaydi, ba'zan yo'q, shuning uchun sinab ko'ramiz.

In [ ]:
train_model['FamilySize'] = train_model['SibSp'] + train_model['Parch'] + 1
train_model['IsAlone'] = (train_model['FamilySize'] == 1).astype(int)

train_model[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

**Muhim eslatma (real tajribadan dars):** Agar `FamilySize`/`IsAlone`ni qo'shib, eski `SibSp`/`Parch`ni saqlab qolsak, model bir xil ma'lumotni ikki marta ko'radi (multicollinearity) — bu birinchi urinishda accuracy'ni 82.1%'dan 80.4%'ga pasaytirdi. Shuning uchun eski, endi ortiqcha bo'lgan ustunlarni olib tashlaymiz.

In [ ]:
X = train_model.drop(['Survived', 'SibSp', 'Parch'], axis=1)
y = train_model['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print("Logistic Regression (FamilySize/IsAlone bilan):", accuracy_score(y_val, y_pred))

**Natija: 81.0%** — dastlabki 82.1%'ga yaqinlashdi, lekin undan hali ham pastroq. Bu shuni ko'rsatadiki: bu kichik va nisbatan oddiy dataset uchun qo'shimcha feature'lar har doim ham foyda bermaydi — "ko'proq feature = yaxshiroq model" degan tasavvur har doim to'g'ri emas.

## 11. Hyperparameter Tuning (GridSearchCV)

Random Forest'ning sozlamalarini (`n_estimators`, `max_depth`, `min_samples_split`) avtomatik ravishda eng yaxshi kombinatsiyasini topish uchun `GridSearchCV` ishlatamiz (5-fold cross-validation bilan).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Eng yaxshi parametrlar:", grid_search.best_params_)

best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_val)
print("Tuned Random Forest accuracy:", accuracy_score(y_val, y_pred_rf))

**Natija: 80.4%** — tuning Random Forest'ni biroz yaxshiladi (79.9% → 80.4%), lekin baribir Logistic Regression'dan (82.1%) pastroq.

---


## 12. Yakuniy xulosa

| Model | Accuracy |
|---|---|
| Logistic Regression (dastlabki) | **82.1%** ✅ eng yaxshi |
| Random Forest (dastlabki) | 79.9% |
| Logistic Regression (FamilySize/IsAlone bilan) | 81.0% |
| Random Forest (tuned: max_depth=5, min_samples_split=5, n_estimators=100) | 80.4% |

### Asosiy xulosalar

1. **"Women and children first" tarixiy qoidasi data bilan tasdiqlandi** — ayollar va bolalarning omon qolish foizi kattalar erkaklarnikidan sezilarli yuqori, va bu Random Forest'ning feature importance tahlilida ham `Sex`ning eng muhim feature ekanini ko'rsatdi.
2. **Eng murakkab model har doim eng yaxshisi emas** — bu nisbatan kichik (891 qator) datada oddiy Logistic Regression murakkabroq Random Forest'dan yaxshiroq natija berdi.
3. **Ko'proq feature qo'shish har doim yaxshilamaydi** — `FamilySize`/`IsAlone` qo'shish natijani pasaytirdi, chunki eski ustunlar bilan ortiqcha (redundant) bo'lib qoldi.
4. **Hyperparameter tuning foydali, lekin sehrli tayoqcha emas** — u Random Forest'ni biroz yaxshiladi, lekin model tanlash strategiyasini almashtira olmadi.

### Keyingi qadamlar (kelajakda qilinishi mumkin)
- `Name` ustunidan unvon (Mr/Mrs/Miss/Master) ajratib olish — bu ko'plab yuqori natijali Titanic yechimlarida qo'llaniladi
- Boshqa algoritmlarni sinab ko'rish (XGBoost, SVM)
- `test.csv` uchun bashorat qilib, Kaggle'ga submission yuborish
